# openSMILE Feature Sets (IEMOCAP)

This notebook extracts standard openSMILE feature sets (eGeMAPS/GeMAPS/ComParE).
Each utterance becomes one training row for downstream SER models.

In [1]:
from pathlib import Path
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

import librosa
import pandas as pd


In [2]:
# Configuration
import sys

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not locate repository root (missing pyproject.toml).")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from feature_extraction.common import machine_name_from_env, resolve_thread_workers

MACHINE_NAME = machine_name_from_env()
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"
OUT_DIR = REPO_ROOT / "extracted_features" / "smilesets"
OUT_FILE = "smilesets_features.csv"

# openSMILE params
FEATURE_SET = "eGeMAPSv02"  # options: GeMAPSv01b, eGeMAPSv02, ComParE_2016

EXCLUDED_EMOTIONS = {"sur", "fea", "oth", "dis"}

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH


PosixPath('/Users/fidjestol/Documents/GitHub/Speech-Emotion-Recognition/extracted_features/smilesets/smilesets_features.csv')

In [3]:
_SMILE_CACHE: dict[str, object] = {}


def _require_opensmile():
    # Import opensmile with a clear error if missing
    try:
        import opensmile
    except ImportError as exc:
        raise ImportError(
            "openSMILE features require the opensmile package. "
            "Install with: uv add opensmile"
        ) from exc
    return opensmile


def _get_smile(feature_set: str):
    # Load or reuse a Smile extractor
    cached = _SMILE_CACHE.get(feature_set)
    if cached is not None:
        return cached

    opensmile = _require_opensmile()
    feature_map = {
        "GeMAPSv01b": opensmile.FeatureSet.GeMAPSv01b,
        "eGeMAPSv02": opensmile.FeatureSet.eGeMAPSv02,
        "ComParE_2016": opensmile.FeatureSet.ComParE_2016,
    }
    if feature_set not in feature_map:
        valid = ", ".join(sorted(feature_map))
        raise ValueError(f"Unknown feature_set '{feature_set}'. Valid: {valid}")

    smile = opensmile.Smile(
        feature_set=feature_map[feature_set],
        feature_level=opensmile.FeatureLevel.Functionals,
    )
    _SMILE_CACHE[feature_set] = smile
    return smile


def extract_smile(audio_path: Path, *, feature_set: str) -> dict[str, float]:
    smile = _get_smile(feature_set)
    features = smile.process_file(str(audio_path))

    flat: dict[str, float] = {}
    for name, value in features.iloc[0].items():
        flat[f"smile_{name}"] = float(value)
    return flat


In [4]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()

# Filter: keep xxx, exclude selected classes, and enforce agreement for labeled classes.
# This keeps unlabeled (xxx) examples while dropping sur/fea/oth/dis.
df = df[~df["emotion"].isin(EXCLUDED_EMOTIONS)].copy()
df = df[(df["emotion"] == "xxx") | (df["agreement"] > 0)].copy()
df.shape


(9887, 7)

In [5]:
CPU_COUNT = os.cpu_count() or 1
COMPUTE_DEVICE = "cpu"  # Force CPU for this CPU-bound extractor
NUM_WORKERS = resolve_thread_workers(MACHINE_NAME)
PROGRESS_MIN_INTERVAL = 1.0

print(
    f"Compute device: {COMPUTE_DEVICE} | extractor_backend=cpu | "
    f"machine={MACHINE_NAME} | workers={NUM_WORKERS}"
)


def process_row(row: dict[str, object]) -> tuple[dict[str, float | str | int] | None, str | None]:
    rel_path = str(row["path"])
    audio_path = AUDIO_ROOT / rel_path
    if not audio_path.exists():
        return None, str(audio_path)

    duration_s = float(librosa.get_duration(filename=str(audio_path)))
    features = extract_smile(audio_path, feature_set=FEATURE_SET)

    record: dict[str, float | str | int] = {
        "path": rel_path,
        "session": int(row["session"]),
        "method": str(row["method"]),
        "gender": str(row["gender"]),
        "emotion": str(row["emotion"]),
        "n_annotators": int(row["n_annotators"]),
        "agreement": int(row["agreement"]),
        "duration_s": float(duration_s),
    }
    record.update(features)
    return record, None


rows: list[dict[str, float | str | int]] = []
missing: list[str] = []
records = df.to_dict(orient="records")

if NUM_WORKERS > 1:
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        mapped = executor.map(process_row, records)
        for record, missing_path in tqdm(mapped, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
            if missing_path is not None:
                missing.append(missing_path)
                continue
            if record is not None:
                rows.append(record)
else:
    for record in tqdm(records, total=len(records), desc="Extracting", unit="file", mininterval=PROGRESS_MIN_INTERVAL):
        row_result, missing_path = process_row(record)
        if missing_path is not None:
            missing.append(missing_path)
            continue
        if row_result is not None:
            rows.append(row_result)

feature_df = pd.DataFrame(rows)
feature_df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
print(f"Workers used: {NUM_WORKERS} (cpu_count={CPU_COUNT})")
if missing:
    print(f"Missing audio files: {len(missing)}")
feature_df.shape


Compute device: cpu | extractor_backend=cpu | machine=macbook | workers=6


Extracting:   0%|          | 0/9887 [00:00<?, ?file/s]

/var/folders/0y/2gsz4yws1zgc_gh4ngp0g29w0000gn/T/ipykernel_66851/3998420809.py:18: FutureWarning: get_duration() keyword argument 'filename' has been renamed to 'path' in version 0.10.0.
	This alias will be removed in version 1.0.
  duration_s = float(librosa.get_duration(filename=str(audio_path)))
/var/folders/0y/2gsz4yws1zgc_gh4ngp0g29w0000gn/T/ipykernel_66851/3998420809.py:18: FutureWarning: get_duration() keyword argument 'filename' has been renamed to 'path' in version 0.10.0.
	This alias will be removed in version 1.0.
  duration_s = float(librosa.get_duration(filename=str(audio_path)))
/var/folders/0y/2gsz4yws1zgc_gh4ngp0g29w0000gn/T/ipykernel_66851/3998420809.py:18: FutureWarning: get_duration() keyword argument 'filename' has been renamed to 'path' in version 0.10.0.
	This alias will be removed in version 1.0.
  duration_s = float(librosa.get_duration(filename=str(audio_path)))
/var/folders/0y/2gsz4yws1zgc_gh4ngp0g29w0000gn/T/ipykernel_66851/3998420809.py:18: FutureWarning: ge

Saved: /Users/fidjestol/Documents/GitHub/Speech-Emotion-Recognition/extracted_features/smilesets/smilesets_features.csv
Workers used: 6 (cpu_count=8)


(9887, 96)